# LAMBADA activation cache

Run on a Colab GPU after saving the training manifest with `dataset.ipynb`. By default, extract
every saved training example and every accepted official validation passage. Test extraction is
optional and disabled by default. Llama and Qwen run sequentially on the same saved contexts.

Each model writes persistent CPU shards as extraction proceeds, so restarting a run reuses
verified completed shards. A combined `paired.pt` is produced for each split in the format used
by `milestone1.ipynb`. Final normalization and LM-head weights are exported while each model is
loaded, together with native-logit reconstruction measurements for baseline evaluation.


In [1]:
%pip install -q \
    "transformers>=5,<6" \
    "huggingface_hub>=1.32,<2" \
    "datasets>=3,<5" \
    "accelerate>=1.0" \
    safetensors \
    tqdm \
    "nltk>=3.9.1,<4" \
    numpy \
    matplotlib


In [2]:
import gc, hashlib, json, os, re
from pathlib import Path
from importlib.metadata import version
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm.auto import tqdm


In [3]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/coms-4705-project")
if not DRIVE_PROJECT_DIR.is_dir():
    raise RuntimeError("Add a My Drive shortcut to the shared coms-4705-project folder.")
print(DRIVE_PROJECT_DIR)

Mounted at /content/drive
/content/drive/MyDrive/coms-4705-project


## Select the saved training set

Leave `TRAIN_DIR` unset if there is exactly one saved manifest; otherwise paste its directory.
`None` limits mean all available accepted examples. `BATCH_SIZE` controls GPU memory use;
`SHARD_SIZE` controls how much work is saved at each checkpoint. Lowering `BATCH_SIZE` after an
out-of-memory error retains completed shards. Keep the shard size and extraction settings fixed
when resuming. Different artifact identities receive separate output directories.


In [4]:
TRAIN_DIR = None
TRAIN_LIMIT = None
VALIDATION_LIMIT = None
INCLUDE_TEST = False
TEST_LIMIT = None
BATCH_SIZE = 8
SHARD_SIZE = 256
ATTENTION_IMPLEMENTATION = "sdpa"
CHECK_ATOL, CHECK_RTOL = 1e-1, 1e-2  # bfloat16 CPU/GPU tolerances, recorded with the measurements.

for label, value in (("TRAIN_LIMIT", TRAIN_LIMIT), ("VALIDATION_LIMIT", VALIDATION_LIMIT), ("TEST_LIMIT", TEST_LIMIT)):
    if value is not None and (type(value) is not int or value <= 0):
        raise ValueError(f"{label} must be None or a positive integer")
if BATCH_SIZE <= 0 or SHARD_SIZE <= 0:
    raise ValueError("Batch and shard sizes must be positive")
if TRAIN_DIR is None:
    manifests = sorted((DRIVE_PROJECT_DIR / "datasets/lambada/train").glob("*/examples.jsonl"))
    if len(manifests) != 1:
        raise ValueError(f"Set TRAIN_DIR to the desired training directory. Found: {manifests}")
    TRAIN_DIR = manifests[0].parent
train_dir = Path(TRAIN_DIR)
meta = json.loads((train_dir / "provenance.json").read_text())
manifest = (train_dir / "examples.jsonl").read_bytes()
if hashlib.sha256(manifest).hexdigest() != meta["manifest_sha256"]:
    raise ValueError("Training manifest checksum mismatch")
training_rows = [json.loads(line) for line in manifest.splitlines()]
if len(training_rows) != meta["n_train"]:
    raise ValueError("Training manifest row count mismatch")
rows_by_split = {"train": training_rows[:TRAIN_LIMIT]}
MODELS, revisions = meta["models"], meta["revisions"]
MAX_TOKENS = meta["max_tokens"]
HIDDEN_SIZES = {"llama": 3072, "qwen": 2048}
if set(MODELS) != set(HIDDEN_SIZES):
    raise ValueError("Expected Llama and Qwen model identities")
print(f"Selected {len(rows_by_split['train']):,} of {len(training_rows):,} saved training examples")


Selected 10,000 of 10,000 saved training examples


In [5]:
import os

from huggingface_hub import hf_hub_download, login, whoami
from huggingface_hub.utils import GatedRepoError

token = os.environ.get("HF_TOKEN")
if token is None:
    from google.colab import userdata
    try:
        token = userdata.get("HF_TOKEN")
    except userdata.SecretNotFoundError as exc:
        raise RuntimeError("Secret 'HF_TOKEN' not found. Add it in the Colab Secrets sidebar.") from exc
    except userdata.NotebookAccessError as exc:
        raise RuntimeError("Notebook access disabled. Toggle on 'Notebook access' for HF_TOKEN.") from exc

login(token=token)
print(f"Authenticated as: {whoami()['name']}")
try:
    hf_hub_download(repo_id=MODELS["llama"], filename="config.json")
except GatedRepoError as exc:
    raise RuntimeError(
        f"Access denied for {MODELS['llama']}. Accept the license on its Hugging Face page."
    ) from exc
print(f"Access confirmed for {MODELS['llama']}")


Authenticated as: edlose


config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

Access confirmed for meta-llama/Llama-3.2-3B


## Label the selected official evaluation splits

Use the saved training run's dataset revision, tokenizer revisions, and context cap. Validation
includes every accepted passage unless a limit is set. The training noun filter is not applied
to evaluation passages. Set `INCLUDE_TEST=True` only when test extraction is intended.


In [6]:
tokenizers = {
    name: AutoTokenizer.from_pretrained(repo, revision=revisions[name])
    for name, repo in MODELS.items()
}
for tok in tokenizers.values():
    tok.padding_side = "right"
    tok.pad_token = tok.eos_token

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/683 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [7]:
WORD = re.compile(r"\S+")
LETTER_WORD = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?")


def split_last_word(text):
    """Prefix, continuation, and word count for a passage whose label is its last word."""
    spans = list(WORD.finditer(text))
    if len(spans) < 2 or spans[-1].end() != len(text):
        return None
    if LETTER_WORD.fullmatch(spans[-1].group()) is None:
        return None
    end = spans[-2].end()
    return text[:end], text[end:], len(spans)

In [8]:
def annotate_tokens(row):
    prefix = row["prefix"]
    full = prefix + row["continuation"]
    for name, tok in tokenizers.items():
        ids = tok.encode(prefix)
        complete = tok.encode(full)
        if len(ids) > MAX_TOKENS or complete[:len(ids)] != ids or len(complete) <= len(ids):
            return None
        row[name + "_length"] = len(ids)
        row[name + "_target_id"] = complete[len(ids)]
        row[name + "_target_length"] = len(complete) - len(ids)
    return row


def label_passage(text, example_id, split):
    parts = split_last_word(text)
    if parts is None:
        return None
    prefix, continuation, n_words = parts
    return annotate_tokens(dict(
        id=example_id, split=split, n_words=n_words,
        prefix=prefix, continuation=continuation,
    ))


def passages_to_rows(split, limit=None):
    rows, dropped = [], 0
    for index, text in enumerate(dataset[split]["text"]):
        row = label_passage(text, f"{split}/{index}", split)
        if row is None:
            dropped += 1
            continue
        rows.append(row)
        if limit is not None and len(rows) >= limit:
            break
    print(f"{split}: {len(rows):,} labeled, {dropped:,} dropped")
    return rows


In [9]:
evaluation_limits = {"validation": VALIDATION_LIMIT}
if INCLUDE_TEST:
    evaluation_limits["test"] = TEST_LIMIT
dataset = {split: load_dataset(meta["dataset"], revision=meta["dataset_revision"], split=split)
           for split in evaluation_limits}
for split, limit in evaluation_limits.items():
    rows_by_split[split] = passages_to_rows(split, limit)
if any(not rows for rows in rows_by_split.values()):
    raise ValueError("Every selected split must contain accepted examples")


README.md:   0%|          | 0.00/7.32k [00:00<?, ?B/s]

plain_text/train-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  269MB            

plain_text/train-00000-of-00002.parquet: downloading bytes:           |  0.00B            

plain_text/train-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  281MB            

plain_text/train-00001-of-00002.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.14MB            

... (3 outputs left out)

Generating train split:   0%|          | 0/2662 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5153 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4869 [00:00<?, ? examples/s]

validation: 4,869 labeled, 0 dropped


## Extract, checkpoint, and export output components

The first extraction batch checks pre/post-normalization consistency. A native-logit comparison
also verifies the exported normalization and output-head weights on a small selection from the
same run; it is integrated into extraction, with no separate pilot run required. Only verified
shards are reused. Each shard records its checksum, IDs, and actual batch size. The final paired
caches retain pre/post-RMSNorm representations and model-specific next-token labels.


In [10]:
@torch.inference_mode()
def extract(name, model, tokenizer, rows, device):
    collected = {"pre": [], "post": []}
    for start in range(0, len(rows), BATCH_SIZE):
        batch = rows[start:start + BATCH_SIZE]
        inputs = tokenizer([row["prefix"] for row in batch], padding=True,
                           return_tensors="pt").to(device)
        lengths = inputs.attention_mask.sum(1)
        assert lengths.tolist() == [row[name + "_length"] for row in batch]
        idx = (torch.arange(len(batch), device=device), lengths - 1)

        def capture(module, args, output):
            collected["pre"].append(args[0][idx].cpu())
            collected["post"].append(output[idx].cpu())

        hook = model.norm.register_forward_hook(capture)
        try:
            out = model(**inputs, use_cache=False)
        finally:
            hook.remove()
        if start == 0:
            post = collected["post"][-1]
            torch.testing.assert_close(post, out.last_hidden_state[idx].cpu())
            torch.testing.assert_close(model.norm(collected["pre"][-1].to(device)).cpu(), post)
        del out
    return {stage: torch.cat(chunks) for stage, chunks in collected.items()}

In [11]:
def save_tensor_file(payload, path):
    from tempfile import NamedTemporaryFile
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with NamedTemporaryFile(dir=path.parent, delete=False) as tmp:
        temp_path = Path(tmp.name)
    try:
        torch.save(payload, temp_path)
        os.replace(temp_path, path)
    finally:
        temp_path.unlink(missing_ok=True)


def save_json(value, path):
    from tempfile import NamedTemporaryFile
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with NamedTemporaryFile(mode="w", encoding="utf-8", dir=path.parent, delete=False) as handle:
        temporary = Path(handle.name)
        json.dump(value, handle, indent=2)
        handle.write("\n")
    try:
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)


def file_sha256(path):
    result = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            result.update(block)
    return result.hexdigest()


def object_sha256(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()


def save_checked_tensor(payload, path):
    path = Path(path)
    save_tensor_file(payload, path)
    save_json({"sha256": file_sha256(path)}, path.with_suffix(".sha256.json"))


def load_checked_tensor(path):
    path = Path(path)
    checksum = path.with_suffix(".sha256.json")
    if not path.is_file() or not checksum.is_file():
        return None
    if json.loads(checksum.read_text())["sha256"] != file_sha256(path):
        return None
    return torch.load(path, map_location="cpu", weights_only=True)


def checked_vectors(path, ids, fingerprint, width):
    payload = load_checked_tensor(path)
    if payload is None:
        return None
    if payload.get("ids") != ids or payload.get("fingerprint") != fingerprint:
        raise ValueError(f"Incompatible IDs/provenance: {path}")
    for stage in ("pre", "post"):
        matrix = payload.get(stage)
        if not isinstance(matrix, torch.Tensor) or matrix.shape != (len(ids), width):
            raise ValueError(f"Invalid tensor shape: {path}/{stage}")
        if not matrix.is_floating_point() or not torch.isfinite(matrix).all():
            raise ValueError(f"Invalid dtype/nonfinite values: {path}/{stage}")
    return payload


def extract_checkpointed(name, backbone, tokenizer, rows, directory, fingerprint, device):
    ids = [row["id"] for row in rows]
    final_path = Path(directory) / f"{name}.pt"
    if checked_vectors(final_path, ids, fingerprint, HIDDEN_SIZES[name]) is not None:
        return final_path
    shards = []
    for start in tqdm(range(0, len(rows), SHARD_SIZE), desc=f"{name}: {Path(directory).name}"):
        batch_rows = rows[start:start + SHARD_SIZE]
        shard_ids = [row["id"] for row in batch_rows]
        path = Path(directory) / "shards" / name / f"{start:08d}.pt"
        payload = checked_vectors(path, shard_ids, fingerprint, HIDDEN_SIZES[name])
        if payload is None:
            if backbone is None:
                raise RuntimeError("A model is required to extract incomplete shards")
            tensors = extract(name, backbone, tokenizer, batch_rows, device)
            if not all(torch.isfinite(v).all() for v in tensors.values()):
                raise ValueError("Nonfinite extracted activations")
            payload = dict(ids=shard_ids, fingerprint=fingerprint, batch_size=BATCH_SIZE, **tensors)
            save_checked_tensor(payload, path)
        shards.append(path)
    payloads = [load_checked_tensor(path) for path in shards]
    # Keeping only final-position vectors bounds CPU storage independently of context lengths.
    combined = dict(ids=ids, fingerprint=fingerprint,
                    pre=torch.cat([p["pre"] for p in payloads]),
                    post=torch.cat([p["post"] for p in payloads]),
                    shard_batch_sizes=[p["batch_size"] for p in payloads])
    save_checked_tensor(combined, final_path)
    return final_path


def reconstruct_head(state, hidden):
    hidden = hidden.to(state["norm_weight"].dtype)
    normalized = hidden.float() * torch.rsqrt(hidden.float().pow(2).mean(-1, keepdim=True) + state["norm_eps"])
    normalized = normalized.to(hidden.dtype) * state["norm_weight"]
    return torch.nn.functional.linear(normalized, state["lm_head_weight"])


def export_and_check_head(name, model, tokenizer, rows, output_dir, identity):
    # Float32 variance and input-dtype casting match these models' RMSNorm operation.
    state = dict(norm_weight=model.model.norm.weight.detach().cpu().clone(),
                 norm_eps=float(model.model.norm.variance_epsilon),
                 lm_head_weight=model.get_output_embeddings().weight.detach().cpu().clone(),
                 hidden_size=model.config.hidden_size, vocab_size=model.config.vocab_size,
                 provenance=identity)
    observations = []
    with torch.inference_mode():
        for row in rows[:3]:
            inputs = tokenizer(row["prefix"], return_tensors="pt").to(model.device)
            captured = {}
            def capture(module, args):
                captured["pre"] = args[0][:, -1, :].detach().cpu().clone()
            hook = model.model.norm.register_forward_pre_hook(capture)
            try:
                native = model(**inputs, use_cache=False).logits[:, -1, :].detach().cpu().float()
            finally:
                hook.remove()
            reconstructed = reconstruct_head(state, captured["pre"]).float()
            close = torch.allclose(native, reconstructed, atol=CHECK_ATOL, rtol=CHECK_RTOL)
            observations.append(dict(id=row["id"], max_abs_error=(native-reconstructed).abs().max().item(),
                                     top1_match=bool(torch.equal(native.argmax(-1), reconstructed.argmax(-1))),
                                     allclose=bool(close)))
    report = dict(provenance=identity, atol=CHECK_ATOL, rtol=CHECK_RTOL, observations=observations,
                  passed=bool(observations) and all(item["allclose"] and item["top1_match"] for item in observations))
    save_json(report, Path(output_dir) / f"{name}_head_validation.json")
    if not report["passed"]:
        raise RuntimeError(f"{name} head reconstruction failed; inspect its validation report")
    head_path = Path(output_dir) / f"{name}_frozen_head.pt"
    save_checked_tensor(state, head_path)
    return head_path


In [12]:
def assemble_cache(split, rows, run_dir, fingerprint, provenance):
    ids = [row["id"] for row in rows]
    cache = dict(
        ids=ids,
        target_token_ids=torch.tensor([row["qwen_target_id"] for row in rows]),
        llama_target_token_ids=torch.tensor([row["llama_target_id"] for row in rows]),
        source_model=MODELS["llama"],
        target_model=MODELS["qwen"],
        provenance=provenance,
    )
    for name, width in HIDDEN_SIZES.items():
        part = torch.load(run_dir / f"{name}.pt", map_location="cpu", weights_only=True)
        assert part["ids"] == ids and part["fingerprint"] == fingerprint
        for stage, suffix in [("pre", ""), ("post", "_post")]:
            matrix = part[stage]
            assert matrix.shape == (len(rows), width) and torch.isfinite(matrix).all()
            cache["h_" + name + suffix] = matrix
    cache_path = run_dir / "paired.pt"
    save_checked_tensor(cache, cache_path)
    saved = torch.load(cache_path, map_location="cpu", weights_only=True)
    assert saved["ids"] == ids
    for key, value in cache.items():
        if isinstance(value, torch.Tensor):
            assert torch.equal(saved[key], value)
            print(key, tuple(value.shape), value.dtype)
    print("Paired cache:", cache_path)
    return cache_path


In [13]:
if not torch.cuda.is_available():
    raise RuntimeError("Select a CUDA GPU runtime for extraction.")
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
environment = {name: version(name) for name in
               ("torch", "transformers", "huggingface_hub", "datasets", "accelerate")}
extraction_settings = dict(schema="paired-full-v1", compute_dtype=str(compute_dtype),
                           attention=ATTENTION_IMPLEMENTATION, shard_size=SHARD_SIZE,
                           padding_side="right", extraction_point="final_token_pre_and_post_rmsnorm")
manifest_data, split_meta, fingerprints = {}, {}, {}
for split, rows in rows_by_split.items():
    ids = [row["id"] for row in rows]
    if len(ids) != len(set(ids)):
        raise ValueError(f"Duplicate IDs in {split}")
    manifest_data[split] = "".join(json.dumps(row, ensure_ascii=False) + "\n" for row in rows).encode()
    info = dict(meta, split=split, n_examples=len(rows),
                manifest_sha256=hashlib.sha256(manifest_data[split]).hexdigest(),
                extraction=extraction_settings, environment=environment)
    split_meta[split] = info
    fingerprints[split] = object_sha256(info)
run_identity = dict(split_fingerprints=fingerprints, models=MODELS, revisions=revisions)
run_hash = object_sha256(run_identity)
counts = "_".join(f"{split}{len(rows)}" for split, rows in rows_by_split.items())
cache_root = DRIVE_PROJECT_DIR / "activations/lambada" / train_dir.name / f"{counts}_{run_hash[:12]}"
cache_root.mkdir(parents=True, exist_ok=True)
save_json(dict(identity=run_identity, settings=extraction_settings, environment=environment), cache_root / "run.json")
for split, rows in rows_by_split.items():
    directory = cache_root / split
    directory.mkdir(parents=True, exist_ok=True)
    manifest_path = directory / "examples.jsonl"
    if manifest_path.exists() and manifest_path.read_bytes() != manifest_data[split]:
        raise ValueError("Output directory contains a different manifest")
    manifest_path.write_bytes(manifest_data[split])
    print(f"{split}: {len(rows):,} examples -> {directory}")

for name, repo in MODELS.items():
    incomplete = []
    for split, rows in rows_by_split.items():
        directory = cache_root / split
        ids = [row["id"] for row in rows]
        if checked_vectors(directory / f"{name}.pt", ids, fingerprints[split], HIDDEN_SIZES[name]) is None:
            incomplete.append(split)
    identity = dict(model=repo, revision=revisions[name], run_hash=run_hash,
                    compute_dtype=str(compute_dtype), attention=ATTENTION_IMPLEMENTATION)
    head_path = cache_root / f"{name}_frozen_head.pt"
    head = load_checked_tensor(head_path)
    head_report_path = cache_root / f"{name}_head_validation.json"
    head_valid = head is not None and head.get("provenance") == identity and head_report_path.is_file()
    if head_valid:
        report = json.loads(head_report_path.read_text())
        head_valid = report.get("provenance") == identity and report.get("passed") is True
    del head
    if not incomplete and head_valid:
        print(f"{name}: verified model caches and output components reused")
        continue
    model = None
    try:
        model = AutoModelForCausalLM.from_pretrained(repo, revision=revisions[name], dtype=compute_dtype,
                                                    attn_implementation=ATTENTION_IMPLEMENTATION)
        model = model.eval().requires_grad_(False).to("cuda")
        if model.config.hidden_size != HIDDEN_SIZES[name]:
            raise ValueError("Model hidden size differs from the cache contract")
        if not head_valid:
            export_and_check_head(name, model, tokenizers[name], rows_by_split["validation"], cache_root, identity)
        for split in incomplete:
            extract_checkpointed(name, model.model, tokenizers[name], rows_by_split[split],
                                 cache_root / split, fingerprints[split], "cuda")
    finally:
        del model
        gc.collect()
        torch.cuda.empty_cache()

paired_paths = {}
for split, rows in rows_by_split.items():
    paired_paths[split] = str(assemble_cache(split, rows, cache_root / split,
                                            fingerprints[split], split_meta[split]))
save_json(dict(identity=run_identity, paired_paths=paired_paths), cache_root / "completed.json")
print("\nUse these settings in milestone1.ipynb:")
print(f"TRAIN_DIR = Path({str(train_dir)!r})")
print(f"CACHE_RUN_DIR = Path({str(cache_root)!r})")


train: 10,000 examples -> /content/drive/MyDrive/coms-4705-project/activations/lambada/7b6847a7f51353d0/train10000_validation4869_a938da615918/train
validation: 4,869 examples -> /content/drive/MyDrive/coms-4705-project/activations/lambada/7b6847a7f51353d0/train10000_validation4869_a938da615918/validation


model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

... (6 outputs left out)

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

qwen: train:   0%|          | 0/40 [00:00<?, ?it/s]

qwen: validation:   0%|          | 0/20 [00:00<?, ?it/s]

target_token_ids (10000,) torch.int64
llama_target_token_ids (10000,) torch.int64
h_llama (10000, 3072) torch.bfloat16
h_llama_post (10000, 3072) torch.bfloat16
h_qwen (10000, 2048) torch.bfloat16
h_qwen_post (10000, 2048) torch.bfloat16
Paired cache: /content/drive/MyDrive/coms-4705-project/activations/lambada/7b6847a7f51353d0/train10000_validation4869_a938da615918/train/paired.pt
target_token_ids (4869,) torch.int64
llama_target_token_ids (4869,) torch.int64
h_llama (4869, 3072) torch.bfloat16
h_llama_post (4869, 3072) torch.bfloat16
h_qwen (4869, 2048) torch.bfloat16
h_qwen_post (4869, 2048) torch.bfloat16
Paired cache: /content/drive/MyDrive/coms-4705-project/activations/lambada/7b6847a7f51353d0/train10000_validation4869_a938da615918/validation/paired.pt

Use these settings in milestone1.ipynb:
TRAIN_DIR = Path('/content/drive/MyDrive/coms-4705-project/datasets/lambada/train/7b6847a7f51353d0')
CACHE_RUN_DIR = Path('/content/drive/MyDrive/coms-4705-project/activations/lambada/7b6847